# Kayak Project — Visualization

This notebook builds the two maps that the brief asks for:

- Top 5 destinations based on the weather score
- Top 20 hotels in those destinations

We use Plotly because the maps are interactive: hovering over a point shows its details.

The next cell imports the libraries and loads the clean files. The output shows 34 cities and 511 hotels.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# Load datasets
cities_weather = pd.read_csv("../data/clean/cities_weather.csv")
hotels_df = pd.read_csv("../data/clean/hotels.csv")

print("Cities:", cities_weather.shape)
print("Hotels:", hotels_df.shape)

Cities: (34, 9)
Hotels: (511, 9)


## Top 5 Destinations

This cell sorts the cities by weather score and keeps the first 5. This is the shortlist for the marketing team. The output shows Bormes les Mimosas, Bayonne, Biarritz, Cassis and Collioure, with scores from 7.82 to 7.12. The ranking depends on the forecasts of 31 October 2026.

In [2]:
top5 = (
    cities_weather
    .sort_values("weather_score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

top5

,city_id,city,latitude,longitude,temperature,humidity,rain_prob,wind_speed,weather_score
0,19,Bormes les Mimosas,43.150697,6.341928,12.19925,70.775,0.16550,2.72550,7.81875
1,34,Bayonne,43.494514,-1.473666,11.36525,81.250,0.06450,3.10000,7.62025
2,33,Biarritz,43.483252,-1.559278,12.09100,78.375,0.06375,3.83425,7.61925
3,20,Cassis,43.214036,5.539632,11.96275,77.450,0.17775,2.61600,7.56925
4,28,Collioure,42.525050,3.083155,12.54800,68.400,0.15625,3.86475,7.12075


## Map — Top 5 Destinations

This cell draws the 5 destinations on a map of France. Color and size both show the weather score. Only 4 points can be told apart at this zoom, because Bayonne and Biarritz are about 7 km apart.

In [3]:
fig = px.scatter_map(
    top5,
    lat="latitude",
    lon="longitude",
    hover_name="city",
    hover_data={
        "temperature": True,
        "humidity": True,
        "rain_prob": True,
        "wind_speed": True,
        "weather_score": True
    },
    color="weather_score",
    size="weather_score",
    zoom=5,
    height=600,
    title="Top 5 Destinations in France (Weather Score)"
)

fig.show()

## Top Hotels Selection

This cell keeps the hotels of the Top 5 cities, removes those without a score or coordinates, sorts them by score and keeps 20. These are the 20 best-rated hotels of the area, across all destinations. Out of 75 hotels, 74 have a score. The output shows the first 5, led by La Recampado at 9.7.

This ranking has limits: it ignores the number of reviews, the tie at 8.9 (5 hotels for 3 places) is settled by the sort order, and Bayonne, the second destination, has no hotel in the Top 20.

In [4]:
top5_cities = top5["city"]

top_hotels = (
    hotels_df[hotels_df["city"].isin(top5_cities)]
    .dropna(subset=["score_clean", "latitude", "longitude"])
    .sort_values("score_clean", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

top_hotels.head()

,city_id,city,hotel_name,score,url,description,latitude,longitude,score_clean
0,19,Bormes les Mimosas,La Recampado,"Avec une note de 9,79,7Exceptionnel 70 expérie...",https://www.booking.com/hotel/fr/la-recampado-...,L’hébergement La Recampado se trouve à Bormes-...,43.150971,6.342117,9.7
1,33,Biarritz,Hôtel Au Bon Coin,"Avec une note de 9,49,4Fabuleux 1 540 expérie...",https://www.booking.com/hotel/fr/au-bon-coin-b...,"Situé à Biarritz, à 400 mètres du front de mer...",43.482174,-1.554190,9.4
2,20,Cassis,La douceur de Cassis,"Avec une note de 9,49,4Fabuleux 98 expérience...",https://www.booking.com/hotel/fr/la-douceur-de...,L’hébergement La douceur de Cassis se trouve à...,43.215009,5.530956,9.4
3,19,Bormes les Mimosas,Les deux mas,"Avec une note de 9,49,4Fabuleux 56 expérience...",https://www.booking.com/hotel/fr/lou-mazet-bor...,"L’établissement Les deux mas se trouve à 3,9 k...",43.143937,6.336323,9.4
4,28,Collioure,Les Roches Brunes,"Avec une note de 9,39,3Fabuleux 658 expérienc...",https://www.booking.com/hotel/fr/les-roches-br...,L’établissement Les Roches Brunes se trouve à ...,42.525348,3.090324,9.3


## Add Jitter to Coordinates

This cell moves each point by a small random offset, up to 0.01 degree, because hotels of the same city overlap on the map. The offset only affects the display, not the data, and it changes at each run. There is no output.

In [5]:
top_hotels["lat_jitter"] = top_hotels["latitude"] + np.random.uniform(-0.01, 0.01, len(top_hotels))
top_hotels["lon_jitter"] = top_hotels["longitude"] + np.random.uniform(-0.01, 0.01, len(top_hotels))

## Scale Marker Size

This cell rescales the hotel scores to marker sizes from 10 to 50. The scores only go from 8.9 to 9.7, so their differences would not be visible without this. There is no output.

In [6]:
top_hotels["score_size"] = (
    (top_hotels["score_clean"] - top_hotels["score_clean"].min()) /
    (top_hotels["score_clean"].max() - top_hotels["score_clean"].min())
) * 40 + 10

## Map — Top 20 Hotels

This cell draws the 20 hotels. Color shows the score and size shows the rescaled score. At the initial zoom, the hotels form 4 groups: Biarritz, Collioure, Cassis and Bormes les Mimosas. Zooming in separates them, and hovering shows the name, the city and the score, along with the shifted coordinates and the rescaled score used for the size.

In [7]:
fig = px.scatter_map(
    top_hotels,
    lat="lat_jitter",
    lon="lon_jitter",
    hover_name="hotel_name",
    hover_data={
        "city": True,
        "score_clean": True
    },
    color="score_clean",
    size="score_size",
    zoom=5,
    height=600,
    title="Top 20 Hotels in Top Destinations"
)

fig.show()

## Figures Cited in the Presentation

The next cells compute the figures used in the presentation, so that each one can be checked here.

This cell compares the Top 5 with the 5 hottest cities. It shows whether the score mostly follows the temperature.

In [8]:
hottest = cities_weather.sort_values("temperature", ascending=False).head(5)["city"]
print("Top 5 by score:", sorted(top5["city"]))
print("5 hottest cities:", sorted(hottest))
print("Same cities:", set(top5["city"]) == set(hottest))

Top 5 by score: ['Bayonne', 'Biarritz', 'Bormes les Mimosas', 'Cassis', 'Collioure']
5 hottest cities: ['Bayonne', 'Biarritz', 'Bormes les Mimosas', 'Cassis', 'Collioure']
Same cities: True


This cell compares the Top 5 of this collection with the Top 5 of the first collection, kept in `data/history/`. The weather changes, so the ranking changes too. It also gives the size of the first collection.

In [9]:
first_weather = pd.read_csv("../data/history/cities_weather_first_collection.csv")
first_hotels = pd.read_csv("../data/history/hotels_first_collection.csv")
first_top5 = first_weather.sort_values("weather_score", ascending=False).head(5)["city"]
print("First collection:", list(first_top5))
print("This collection:", list(top5["city"]))
print("Kept:", sorted(set(top5["city"]) & set(first_top5)))
print("New destinations:", sorted(set(top5["city"]) - set(first_top5)))
print("First collection, hotels per city:", first_hotels["city"].value_counts().unique(), "| total:", len(first_hotels))

First collection: ['Aix en Provence', 'Paris', 'Cassis', 'Bormes les Mimosas', 'Marseille']
This collection: ['Bormes les Mimosas', 'Bayonne', 'Biarritz', 'Cassis', 'Collioure']
Kept: ['Bormes les Mimosas', 'Cassis']
New destinations: ['Bayonne', 'Biarritz', 'Collioure']
First collection, hotels per city: [25] | total: 125


This cell measures how much each term of the score varies between the 34 cities, and shows the last city of the ranking. It shows which term separates the cities the most.

In [10]:
terms = pd.DataFrame({
    "temperature": cities_weather["temperature"],
    "wind_speed": cities_weather["wind_speed"],
    "rain_prob * 10": cities_weather["rain_prob"] * 10,
})
print(terms.agg(["min", "max"]).T.assign(gap=lambda t: t["max"] - t["min"], std=terms.std()).round(2))
print("Humidity, min and max:", round(cities_weather["humidity"].min(), 1), round(cities_weather["humidity"].max(), 1))
print(cities_weather.sort_values("weather_score").head(1)[["city", "weather_score"]].round(2).to_string(index=False))

                 min    max    gap   std
temperature     2.10  12.55  10.45  2.69
wind_speed      0.82   5.86   5.04  1.22
rain_prob * 10  0.00   2.42   2.42  0.77
Humidity, min and max: 68.4 86.7
                        city  weather_score
Chateau du Haut Koenigsbourg          -0.59


This cell describes the Top 20: number of experiences behind each score, hotels per city and distance to the city. It also counts the hotels available for this ranking, and the tie at the last score.

In [11]:
def distance_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


experiences = top_hotels["score"].str.extract(r"(\d[\d\s]*) expériences?")[0].str.replace(r"\s", "", regex=True).astype(int)
with_city = top_hotels.merge(cities_weather[["city", "latitude", "longitude"]], on="city", suffixes=("", "_city"))
top_km = distance_km(with_city["latitude"], with_city["longitude"], with_city["latitude_city"], with_city["longitude_city"])

in_top5 = hotels_df[hotels_df["city"].isin(top5_cities)]
candidates = in_top5.dropna(subset=["score_clean", "latitude", "longitude"])
last_score = top_hotels["score_clean"].min()
print("Experiences, first hotel, min and max:", experiences.iloc[0], experiences.min(), experiences.max())
print("Hotels per city:", top_hotels["city"].value_counts().to_dict())
print("Bayonne hotels in the Top 20:", (top_hotels["city"] == "Bayonne").sum())
print("Largest distance to the city (km):", round(top_km.max(), 1))
print("Hotels in the Top 5 cities:", len(in_top5), "| with a score and coordinates:", len(candidates))
print(f"Hotels at {last_score}:", (candidates["score_clean"] == last_score).sum(), "| kept in the Top 20:", (top_hotels["score_clean"] == last_score).sum())

Experiences, first hotel, min and max: 70 38 1540
Hotels per city: {'Bormes les Mimosas': 6, 'Biarritz': 6, 'Cassis': 6, 'Collioure': 2}
Bayonne hotels in the Top 20: 0
Largest distance to the city (km): 3.5
Hotels in the Top 5 cities: 75 | with a score and coordinates: 74
Hotels at 8.9: 5 | kept in the Top 20: 3


This cell describes the scores of all the hotels, and counts how many hotels are far from their city. It also measures the distance between Bayonne and Biarritz, which overlap on the Top 5 map.

In [12]:
scores = hotels_df["score_clean"]
print("Hotels:", len(hotels_df), "| scores read:", scores.notna().sum(), "| min:", scores.min(), "| median:", scores.median(), "| max:", scores.max())
print("Distinct descriptions:", hotels_df["description"].nunique())

all_with_city = hotels_df.merge(cities_weather[["city", "latitude", "longitude"]], on="city", suffixes=("", "_city"))
all_km = distance_km(all_with_city["latitude"], all_with_city["longitude"], all_with_city["latitude_city"], all_with_city["longitude_city"])
print("Hotels farther than 10, 25 and 50 km:", (all_km > 10).sum(), (all_km > 25).sum(), (all_km > 50).sum())

pair = cities_weather.set_index("city").loc[["Bayonne", "Biarritz"]]
print("Bayonne to Biarritz (km):", round(distance_km(pair["latitude"].iloc[0], pair["longitude"].iloc[0], pair["latitude"].iloc[1], pair["longitude"].iloc[1]), 1))

Hotels: 511 | scores read: 510 | min: 5.9 | median: 8.4 | max: 10.0
Distinct descriptions: 511
Hotels farther than 10, 25 and 50 km: 28 3 0
Bayonne to Biarritz (km): 7.0


This cell reads the duration of notebook 02 from the progress log of the collection.

In [13]:
log = pd.read_csv("../data/raw/collection_progress.log", sep=" \\| ", engine="python", header=None, usecols=[0, 1], names=["time", "step"])
log["time"] = pd.to_datetime(log["time"])
start = log.loc[log["step"] == "START 02_booking_scraping", "time"].iloc[0]
end = log.loc[log["step"] == "DONE 02_booking_scraping", "time"].iloc[0]
print("Duration of notebook 02:", end - start)

Duration of notebook 02: 0 days 01:54:46
